In [1]:
import pandas as pd
import numpy as np

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

In [2]:
boundaries_df = pd.read_csv("/scratch1/smaruj/natural_boundaries_URQmean/all_natural_boundaries.tsv", sep="\t")

In [ ]:
boundaries_df = pd.read_csv("/home1/smaruj/akitaV2-analyses/experiments/TAD_boundaries_analysis/input_data/TAD_boundaries/all_boundaries.tsv", sep="\t")
boundaries_df = boundaries_df.loc[:, ~boundaries_df.columns.str.contains('^Unnamed')]

In [ ]:
boundaries_df

In [ ]:
chrom_sizes = "/project/fudenber_735/genomes/mm10/mm10.fa.sizes"

In [ ]:
chrom_sizes_table = pd.read_csv(chrom_sizes, sep="\t", names=["chrom", "size"])

In [ ]:
# all the TAD boundaries are 10kb-long
# for the comparisson purposes, let's center all the prediction windows on the TAD boundaries

seq_length = 1310720

In [ ]:
rel_tad_start = int((((seq_length / 2) - 5000) // 2048) * 2048)

In [ ]:
# let's approximate 4.88 bins as 5 bins

rel_tad_end = rel_tad_start + 2048*5

In [ ]:
to_window_end = seq_length - rel_tad_start

In [ ]:
boundaries_df["window_start"] = boundaries_df["start"] - rel_tad_start
boundaries_df["window_end"] = boundaries_df["start"] + to_window_end

In [ ]:
# double checking if we didn't get outside of chromosomes
len(boundaries_df[boundaries_df["window_start"] <= 0])

In [ ]:
check_chrom = boundaries_df.groupby(by="chrom")["window_end"].max().reset_index()

In [ ]:
for index, row in check_chrom.iterrows():
    chr_size = int(
        chrom_sizes_table.loc[
            chrom_sizes_table["chrom"] == row.chrom, "size"
        ].iloc[0]
    )
    if chr_size < row.window_end:
        print("problem")
    

In [ ]:
boundaries_df

In [ ]:
# boundaries_df.to_csv("/scratch1/smaruj/natural_boundaries_URQmean/all_natural_boundaries.tsv", sep="\t", index=False)

In [3]:
test = boundaries_df[:10]

In [4]:
test

,chrom,start,end,region,is_bad_bin,log2_insulation_score_200000,n_valid_pixels_200000,boundary_strength_200000,is_boundary_200000,window_start,window_end
0,chr1,4400000,4410000,chr1,False,-0.425816,358.0,1.013374,True,3750784,5061504
1,chr1,4770000,4780000,chr1,False,-0.489156,397.0,1.002922,True,4120784,5431504
2,chr1,5150000,5160000,chr1,False,-0.231962,377.0,0.498138,True,4500784,5811504
3,chr1,5900000,5910000,chr1,False,-0.404405,397.0,0.873738,True,5250784,6561504
4,chr1,6190000,6200000,chr1,False,-0.452379,397.0,0.964178,True,5540784,6851504
5,chr1,6430000,6440000,chr1,True,-0.387889,342.0,0.391166,True,5780784,7091504
6,chr1,6980000,6990000,chr1,False,-0.450711,397.0,0.948860,True,6330784,7641504
7,chr1,7230000,7240000,chr1,True,-0.315994,342.0,0.563464,True,6580784,7891504
8,chr1,8960000,8970000,chr1,False,-0.046662,357.0,0.279394,True,8310784,9621504
9,chr1,9540000,9550000,chr1,False,-0.555013,397.0,1.080191,True,8890784,10201504


In [5]:
import random

In [6]:
from torch.utils.data import Dataset, DataLoader
import torch

In [7]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN


In [8]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return one_hot_encoded

In [9]:
class GenomicSequenceDataset(Dataset):
    def __init__(self, coord_df, genome_fasta):
        self.coords = coord_df  # DataFrame with chrom, start, end
        self.genome = genome_fasta

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        TARGET_LEN = 1310720
        
        row = self.coords.iloc[idx]
        chrom, start, end = row["chrom"], row["window_start"], row["window_end"]
        seq = self.genome[chrom][start:end].seq.upper()
        
        # Fix sequence length if needed
        if len(seq) != TARGET_LEN:
            seq = seq[:TARGET_LEN].ljust(TARGET_LEN, 'N')  # pad with Ns if needed
        
        one_hot = one_hot_encode_sequence(seq)  # shape: (4, L)
        return torch.from_numpy(one_hot.copy())

In [10]:
def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = batch_vectors.shape[0]
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512]  

In [11]:
from pyfaidx import Fasta

In [12]:
fasta_file = "/project/fudenber_735/genomes/hg38/hg38.fa"
genome = Fasta(fasta_file)

In [13]:
orig_dataset = GenomicSequenceDataset(test, genome)

In [14]:
orig_loader = DataLoader(orig_dataset, batch_size=10, shuffle=False)

In [15]:
# --- Load model ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_927631/ipykernel_2748183/2795485947.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_f

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [16]:
URQ_mean_all = []

with torch.no_grad():
    for orig_batch in orig_loader:
        orig_preds = model(orig_batch.to(device)).cpu()
        
        maps = from_upper_triu_batch(orig_preds)
        URQ_mean = np.nanmean(maps[:, 0:250, 260:512], axis=(1, 2))  # shape: [B]
        
        URQ_mean_all.extend(URQ_mean)

In [17]:
test["URQ_mean"] = URQ_mean_all

/tmp/SLURM_927631/ipykernel_2748183/4175067186.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["URQ_mean"] = URQ_mean_all


In [18]:
test

,chrom,start,end,region,is_bad_bin,log2_insulation_score_200000,n_valid_pixels_200000,boundary_strength_200000,is_boundary_200000,window_start,window_end,URQ_mean
0,chr1,4400000,4410000,chr1,False,-0.425816,358.0,1.013374,True,3750784,5061504,-0.046626
1,chr1,4770000,4780000,chr1,False,-0.489156,397.0,1.002922,True,4120784,5431504,-0.097691
2,chr1,5150000,5160000,chr1,False,-0.231962,377.0,0.498138,True,4500784,5811504,-0.034787
3,chr1,5900000,5910000,chr1,False,-0.404405,397.0,0.873738,True,5250784,6561504,-0.269369
4,chr1,6190000,6200000,chr1,False,-0.452379,397.0,0.964178,True,5540784,6851504,-0.229362
5,chr1,6430000,6440000,chr1,True,-0.387889,342.0,0.391166,True,5780784,7091504,-0.174728
6,chr1,6980000,6990000,chr1,False,-0.450711,397.0,0.948860,True,6330784,7641504,-0.137547
7,chr1,7230000,7240000,chr1,True,-0.315994,342.0,0.563464,True,6580784,7891504,-0.121672
8,chr1,8960000,8970000,chr1,False,-0.046662,357.0,0.279394,True,8310784,9621504,-0.279851
9,chr1,9540000,9550000,chr1,False,-0.555013,397.0,1.080191,True,8890784,10201504,-0.139804


In [19]:
test.to_csv("/scratch1/smaruj/natural_boundaries_URQmean/test.tsv", sep="\t", index=False)